# TEXL - Tamper-Evident eXplanation Ledger: Proof-of-Concept Notebook

**AML Lab (CSE2512P) - Track 1: Patent Design - Team: Smooth operators - IILM University, Gurugram**

This notebook is the proof of concept (PoC) for the invention disclosed in the report. It shows, step by step,
how every claim of the claims set is realised in working code:

| Claim | What it covers | Section below |
|---|---|---|
| 1 (independent) | seeded model-agnostic explanation, committed input, canonical record, hash chain, signature, receipt | 3-5 |
| 2 | additivity (local-accuracy) gate before a record is written | 6 |
| 3 | Merkle-sealed epochs, external witness, inclusion proof in the receipt | 5 |
| 4 | deterministic replay verification of score and explanation | 8 |
| 5 | fixed-point canonical serialisation | 9 |
| 6 | adverse-action reasons cryptographically bound to the logged explanation | 4 |

Data: UCI Statlog (German Credit Data), CC BY 4.0, DOI 10.24432/C5NC77. All random seeds are fixed.

In [1]:
import sys, json, copy, hashlib
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd
from texl import canonical, data, models
from texl.explain import PermutationShapley
from texl.ledger import Ledger, demo_signing_key, ConsistencyError, additivity_residual
from texl.verify import verify_ledger, verify_receipt, Replayer
from texl.attacks import resign_from
pd.set_option("display.width", 140)
SEED = 42

## 1. Data and leakage-free split (70 / 15 / 15, stratified, seed 42)

In [2]:
X, y = data.load()
print("rows:", len(X), " features:", X.shape[1], " bad-credit rate:", round(y.mean(), 3))
print("SHA-256 of german.data:", hashlib.sha256(data.RAW_FILE.read_bytes()).hexdigest())
X_tr, y_tr, X_va, y_va, X_te, y_te = data.split(X, y, seed=SEED)
print("train/val/test:", len(X_tr), len(X_va), len(X_te))
X.head()

rows: 1000  features: 20  bad-credit rate: 0.3
SHA-256 of german.data: b21f3d81db8071257d5ff1deaeba1fd4303b62712e6fcc9715c7a86202cb5871
train/val/test: 700 150 150


,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,personal_status,other_parties,residence_since,property_magnitude,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker
0,A11,6.0,A34,A43,1169.0,A65,A75,4.0,A93,A101,4.0,A121,67.0,A143,A152,2.0,A173,1.0,A192,A201
1,A12,48.0,A32,A43,5951.0,A61,A73,2.0,A92,A101,2.0,A121,22.0,A143,A152,1.0,A173,1.0,A191,A201
2,A14,12.0,A34,A46,2096.0,A61,A74,2.0,A93,A101,3.0,A121,49.0,A143,A152,1.0,A172,2.0,A191,A201
3,A11,42.0,A32,A42,7882.0,A61,A74,2.0,A93,A103,4.0,A122,45.0,A143,A153,1.0,A173,2.0,A191,A201
4,A11,24.0,A33,A40,4870.0,A61,A73,3.0,A93,A101,4.0,A124,53.0,A143,A153,2.0,A173,2.0,A191,A201


## 2. Credit model (any black box works - here Gradient Boosting)
Pre-processing (scaling, one-hot) is fitted **inside** the pipeline on the training split only. The decision threshold
minimises the Statlog cost matrix (approving a bad applicant costs 5, rejecting a good one costs 1) on the validation split.

In [3]:
model = models.make_pipeline("gbm", SEED).fit(X_tr, y_tr)
thr = models.choose_threshold(y_va, model.predict_proba(X_va)[:, 1])
print("cost-optimal threshold on validation:", thr)
print("test metrics:", {k: round(v, 3) for k, v in models.evaluate(y_te, model.predict_proba(X_te)[:, 1], thr).items()})
model_fp = models.model_fingerprint(model)
print("model fingerprint F(f):", model_fp)

cost-optimal threshold on validation: 0.1
test metrics: {'roc_auc': 0.816, 'pr_auc': 0.688, 'brier': 0.15, 'accuracy': 0.573, 'cost': 0.533, 'reject_rate': 0.673}
model fingerprint F(f): edf791723ac6ecf3c580a36b2450683771d40ab3b24867abe2abe46695962a67


## 3. Seeded, model-agnostic permutation-Shapley explainer (Claim 1c)
The explainer only calls `f(x) = P(bad | x)`. Its configuration (seed, permutations, background set) is hashed into a
**configuration digest**; the background set is bound by its own digest, so it cannot be swapped silently.

In [4]:
codec = data.RawCodec(X_tr)
rng = np.random.default_rng(SEED)
background = codec.encode(X_tr.iloc[rng.choice(len(X_tr), 32, replace=False)])
f = lambda Z: model.predict_proba(codec.decode(Z))[:, 1]
explainer = PermutationShapley(f, background, data.FEATURES, n_permutations=16, seed=2026)
print(json.dumps({k: v for k, v in explainer.config.items() if k != "features"}, indent=1))
print("configuration digest:", explainer.config_digest)

{
 "method": "perm-shapley-antithetic/v1",
 "n_permutations": 16,
 "seed": 2026,
 "background_size": 32,
 "background_digest": "6a990b454ffb84fb0d510ddb6330b755f86a42d0d91baf4823ad9ce890bb73a6",
 "output": "P(bad)",
 "scale": 1000000
}
configuration digest: d5ea824412aa97a96c09ac944939cd3303fd9c05425246db618bca5857acb0ae


## 4. One explained, logged decision (Claims 1 and 6)

In [5]:
sk = demo_signing_key()           # deterministic demo key - a real lender would use an HSM
ledger = Ledger(sk, model_fp, explainer.config_digest, data.FEATURES, thr, epoch_size=32)
Xenc = codec.encode(X_te)
salts = [np.random.default_rng(1000 + i).bytes(16) for i in range(len(Xenc))]  # demo salts (use os.urandom in production)
input_store = {}
rec0 = ledger.decide(Xenc[0], explainer, salt=salts[0])
input_store[rec0["body"]["input_commitment"]] = (salts[0], Xenc[0])
b = rec0["body"]
print("decision:", b["decision"], "  P(bad) =", b["score"] / 1e6, "  reasons:", b["reasons"])
print("additivity residual (units of 1e-6):", additivity_residual(b))
print(json.dumps(rec0, indent=1)[:1500], "...")

decision: APPROVE   P(bad) = 0.059449   reasons: ['employment', 'personal_status', 'age']
additivity residual (units of 1e-6): 1
{
 "body": {
  "v": 1,
  "seq": 0,
  "ts": "2026-11-01T09:00:00Z",
  "model_fp": "edf791723ac6ecf3c580a36b2450683771d40ab3b24867abe2abe46695962a67",
  "explainer_cfg_digest": "d5ea824412aa97a96c09ac944939cd3303fd9c05425246db618bca5857acb0ae",
  "input_commitment": "e6ca22763df51880529df2c634da0f4885269675f207702de2679be2b22bad1f",
  "score": 59449,
  "threshold": 100000,
  "decision": "APPROVE",
  "base_value": 419172,
  "attributions": {
   "checking_status": -116960,
   "duration": -27827,
   "credit_history": -2805,
   "purpose": -80017,
   "credit_amount": -74147,
   "savings_status": -11309,
   "employment": 44171,
   "installment_commitment": -33057,
   "personal_status": 6571,
   "other_parties": 551,
   "residence_since": -3005,
   "property_magnitude": -3764,
   "age": 5501,
   "other_payment_plans": -40512,
   "housing": -14668,
   "existing_credits

## 5. Ledger for the whole test set, epoch sealing and applicant receipts (Claims 1g and 3)

In [6]:
for i in range(1, len(Xenc)):
    r = ledger.decide(Xenc[i], explainer, salt=salts[i])
    input_store[r["body"]["input_commitment"]] = (salts[i], Xenc[i])
ledger.seal_epoch()
cps = ledger.witness.checkpoints
print(len(ledger.records), "records,", len(cps), "checkpoints published to the external witness")
print("first checkpoint:", json.dumps(cps[0]["body"], indent=1))
rc = ledger.receipts[5]
print("receipt for seq 5 -> inclusion proof length:", len(rc["proof"]))
print("applicant-side verification:", verify_receipt(rc, ledger.public_key, cps, canonical.fixed_vector(Xenc[5])))

150 records, 5 checkpoints published to the external witness
first checkpoint: {
 "epoch": 0,
 "first_seq": 0,
 "tree_size": 32,
 "root": "229a41716b4f17594c8e29d7a2e3fb57ab7c806a65d54051c5eb15065c2ecab7",
 "prev_checkpoint": "0000000000000000000000000000000000000000000000000000000000000000",
 "ts": "2026-11-01T09:00:32Z"
}
receipt for seq 5 -> inclusion proof length: 5
applicant-side verification: {'record_hash_ok': True, 'signature_ok': True, 'additivity_ok': True, 'commitment_ok': True, 'inclusion_ok': True, 'checkpoint_published': True, 'valid': True}


## 6. Additivity gate rejects an inconsistent explanation (Claim 2)

In [7]:
demo = Ledger(sk, model_fp, explainer.config_digest, data.FEATURES, thr)
score, phi = explainer.explain(Xenc[1])
xf = canonical.fixed_vector(Xenc[1])
body = demo.build_body(xf, salts[1], score, phi, explainer.base_value)
body["attributions"]["personal_status"] = 0       # try to hide a protected attribute
try:
    demo.append(body, salts[1], xf)
except ConsistencyError as e:
    print("BLOCKED by gate:", e)

BLOCKED by gate: explanation does not reproduce score for seq 0


## 7. Regulator verification of the honest ledger - no false alarms

In [8]:
replayer = Replayer(model_fp, explainer, input_store, thr)
findings = verify_ledger(ledger.records, ledger.public_key, witness_checkpoints=cps,
                         receipts=list(ledger.receipts.values())[:8], replayer=replayer)
print("findings on honest ledger:", findings)

findings on honest ledger: []


## 8. Tampering demonstrations (Claims 3 and 4)
(a) An **insider** flips a decision and re-signs the whole tail: the hash chain and signatures look valid, but the
anchored Merkle checkpoint exposes the rewrite.  (b) An insider **fairwashes** an explanation *at logging time*,
moving the protected-attribute contribution to another feature so that additivity still holds: only deterministic
replay detects it.

In [9]:
recs = copy.deepcopy(ledger.records)
recs[10]["body"]["decision"] = "APPROVE" if recs[10]["body"]["decision"] == "REJECT" else "REJECT"
resign_from(recs, 10, sk)
print("chain+signature checks:", verify_ledger(recs, ledger.public_key, checks=["hash", "chain", "sig"]))
print("anchor check:", verify_ledger(recs, ledger.public_key, checks=["anchor"], witness_checkpoints=cps)[:2])

chain+signature checks: []
anchor check: [{'check': 'anchor', 'seq': 0, 'detail': 'Merkle root differs from published checkpoint'}, {'check': 'anchor', 'seq': 32, 'detail': 'Merkle root differs from published checkpoint'}]


In [10]:
t = int(np.argmax([abs(sum(r["body"]["attributions"][p] for p in data.PROTECTED)) for r in ledger.records]))
recs = copy.deepcopy(ledger.records)
att = recs[t]["body"]["attributions"]
moved = sum(att[p] for p in data.PROTECTED)
for p in data.PROTECTED: att[p] = 0
att["checking_status"] += moved                     # sum preserved -> additivity gate passes
resign_from(recs, t, sk)                            # done at birth: receipts/checkpoints would agree
no_replay = ["hash", "chain", "sig", "additivity"]
print("without replay:", verify_ledger(recs, ledger.public_key, checks=no_replay))
print("with replay   :", [f for f in verify_ledger(recs, ledger.public_key, checks=["replay"], replayer=replayer)])

without replay: []
with replay   : [{'check': 'replay', 'seq': 133, 'detail': 'replayed attribution differs by 132438 units'}]


## 9. Why fixed-point canonicalisation is needed (Claim 5)

In [11]:
s1, p1 = explainer.explain(Xenc[3])
s2, p2 = explainer.explain(Xenc[3], reverse_accumulation=True)   # same maths, different float summation order
print("max |difference| between float runs:", np.max(np.abs(p1 - p2)))
print("raw float64 JSON hashes equal? ", hashlib.sha256(canonical.naive_float_bytes(p1.tolist())).hexdigest()
      == hashlib.sha256(canonical.naive_float_bytes(p2.tolist())).hexdigest())
print("fixed-point vectors equal?     ", canonical.fixed_vector(p1) == canonical.fixed_vector(p2))

max |difference| between float runs: 2.42861286636753e-17
raw float64 JSON hashes equal?  False
fixed-point vectors equal?      True


## 10. Full evaluation results (produced by `scripts/run_experiments.py`)

In [12]:
res = json.loads((ROOT / "results" / "results.json").read_text())
det = pd.DataFrame(res["tamper"]["detection_pct"]).T
det.insert(0, "attack", [res["tamper"]["attacks"][a] for a in det.index])
det

,attack,V0 Plain log,V1 +Record hash,V2 +Hash chain,V3 +Signatures,V4 +Anchored checkpoints,V5 +Applicant receipts,V6 +Additivity gate,V7 TEXL (+Replay)
A1,"Edit one attribution, no re-hash (outsider)",0.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0
A2,"Flip decision, re-hash tail, no key (outsider)",0.0,0.0,0.0,100.0,100.0,100.0,100.0,100.0
A3,"Flip decision, re-hash + re-sign tail (insider)",0.0,0.0,0.0,0.0,84.0,90.0,90.0,100.0
A4,"Delete a record, re-link + re-sign (insider)",0.0,0.0,0.0,0.0,87.5,94.5,94.5,94.5
A5,"Swap two adjacent records, re-sign (insider)",0.0,0.0,0.0,0.0,80.5,89.5,89.5,89.5
A6,Truncate the last 5 records (insider),0.0,0.0,0.0,0.0,0.0,21.5,21.5,21.5
A7,Fairwash at birth: zero protected attributions...,0.0,0.0,0.0,0.0,0.0,0.0,100.0,100.0
A8,"Fairwash at birth: move protected mass, sum ke...",0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0
A9,Undisclosed model substitution at birth (insider),0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0


Insider deletion, reordering and truncation can only succeed in the **unsealed tail** (records not yet covered by a
published checkpoint). The sealing-window experiment re-runs attacks A3–A6 with smaller epochs:

In [13]:
pd.read_csv(ROOT / "results" / "tables" / "sealing_window.csv")

,epoch_size,unsealed_tail,checkpoints,A3 V4,A3 V5,A3 V7,A4 V4,A4 V5,A4 V7,A5 V4,A5 V5,A5 V7,A6 V4,A6 V5,A6 V7,false_alarm_V7
0,32,22,4,84.0,90.0,100.0,87.5,94.5,94.5,80.5,89.5,89.5,0.0,21.5,21.5,0.0
1,8,6,18,97.0,98.0,100.0,97.0,97.0,97.0,97.5,98.0,98.0,0.0,21.5,21.5,0.0
2,1,0,150,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,0.0


In [14]:
pd.DataFrame({k: {m: round(v[0], 3) for m, v in res["models"][k].items()} for k in ["logreg", "rf", "gbm"]})

,logreg,rf,gbm
roc_auc,0.787,0.796,0.793
pr_auc,0.599,0.623,0.609
brier,0.167,0.166,0.164
accuracy,0.609,0.584,0.585
cost,0.529,0.560,0.537
reject_rate,0.621,0.644,0.653
threshold,0.156,0.222,0.152


In [15]:
pd.DataFrame(res["overhead"]).T[["predict_ms", "explain_ms", "ledger_append_ms", "verify_per_record_ms",
                                 "receipt_verify_ms", "record_bytes", "honest_findings"]]

,predict_ms,explain_ms,ledger_append_ms,verify_per_record_ms,receipt_verify_ms,record_bytes,honest_findings
logreg,"[5.178625, 0.838521]","[25.875499, 6.869709]","[0.269873, 0.086775]",0.170554,0.189906,1097.72,0
rf,"[17.226541, 6.248085]","[137.733813, 25.025036]","[0.290451, 0.125764]",0.17205,0.183497,1086.18,0
gbm,"[6.070476, 2.645736]","[41.221635, 13.601394]","[0.287949, 0.106484]",0.177544,0.185334,1088.066667,0
